# autoresearch-medimage — Colab Quick Start

Run a complete search → train → report cycle on a free Colab GPU.

**Requirements:** Colab with GPU runtime (T4 free tier works).

This notebook assumes the repo is public. It pulls a public source snapshot into `/content`, so users do not need GitHub tokens, SSH keys, or private-repo access.

**Fastest public trial path:** run Setup, run the Smoke Test, then upload your own dataset zip.

For the full ChestX-ray14 `chexnet` demo from a local dataset copy, use `notebooks/colab_chexnet.ipynb`.

| GPU | VRAM | ~Experiments/hour | Notes |
|-----|------|-------------------|-------|
| T4 (free) | 16 GB | ~10-12 | Free tier, ~4 hrs/day |
| L4 (Pro) | 24 GB | ~12-14 | Larger batch sizes |
| A100 (Pro+) | 40 GB | ~12-14 | Fastest, biggest models |

## 1. Setup

This setup path is public and no-auth. It downloads a snapshot of `main` into `/content/autoresearch-medimage` so anyone can try the repo from their own Colab runtime.

In [ ]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Install uv (fast Python package manager)
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

In [ ]:
# Download the public repo snapshot into /content (no GitHub token needed)
!rm -rf /content/autoresearch-medimage /content/autoresearch-medimage-main /content/repo.zip
!wget -q https://github.com/mlungrengmail/autoresearch-medimage/archive/refs/heads/main.zip -O /content/repo.zip
!unzip -q /content/repo.zip -d /content
!mv /content/autoresearch-medimage-main /content/autoresearch-medimage
%cd /content/autoresearch-medimage

In [ ]:
# Install dependencies (first run takes ~2 min, cached after)
!uv sync

## 2. Smoke Test (no download, no GPU needed)

This is the fastest public trial run. It uses the built-in toy dataset and confirms the Colab runtime can install dependencies, prepare data, and train successfully.

In [ ]:
!uv run prepare.py --dataset toy_binary
!uv run train.py --recipe toy_binary_smoke --time-budget 5

## 3. CheXNet Demo - Single Experiment

Optional heavier demo after the smoke test passes.
If you already have ChestX-ray14 locally, use `notebooks/colab_chexnet.ipynb` instead.
The cells below are still the Kaggle-backed path.

In [ ]:
# Set Kaggle credentials (get yours from kaggle.com/settings)
import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
# Uncomment and fill in:
# os.environ["KAGGLE_USERNAME"] = "your_username"
# os.environ["KAGGLE_KEY"] = "your_key"

In [ ]:
!uv run prepare.py --dataset chestxray14

In [ ]:
# Run one 5-minute baseline experiment
!uv run train.py --recipe chexnet --save-checkpoint

In [ ]:
# Finalize: held-out test eval, paper, figures
!uv run python tools/finalize_run.py --dataset chestxray14

## 4. Your Own Dataset

This is the easiest no-auth public path for trying your own data.
Upload a zip of your images to Colab, or mount Google Drive.

In [ ]:
# Option A: Upload a zip
# from google.colab import files
# uploaded = files.upload()  # upload your_dataset.zip
# !unzip -q your_dataset.zip -d /content/my_images

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/my_images'

In [ ]:
# Auto-detect and prepare your dataset
# !uv run prepare.py --data /content/my_images

# Train on it
# !uv run train.py --dataset my_images

## 5. Multi-Experiment Search Session

For longer search sessions, use `--session` to isolate experiments.
On the free T4 tier (~4 hrs/day), expect 40-50 experiments per session.

In [ ]:
SESSION = "colab_search_001"

# Baseline experiment
!uv run train.py --recipe chexnet --session {SESSION} --save-checkpoint > run.log 2>&1
!tail -20 run.log
!uv run python tools/mark_result.py --dataset chestxray14 --session {SESSION} --status keep

In [ ]:
# Check results so far
import pandas as pd
df = pd.read_csv("results.tsv", sep="\t")
if "session" in df.columns:
    df = df[df["session"] == SESSION]
df[["timestamp", "backbone", "metric", "status"]].tail(10)

In [ ]:
# Finalize the session
!uv run python tools/finalize_run.py --dataset chestxray14 --session {SESSION}

## 6. Resume After Disconnect

Colab sessions disconnect after idle. To resume:

1. Re-run the Setup cells (clone + uv sync)
2. Your `data/` directory is gone — re-run `prepare.py` or mount Drive
3. `results.tsv` and checkpoints are also gone unless saved to Drive

**Tip:** For longer sessions, save results to Google Drive:

In [ ]:
# Save results to Drive (run periodically during long sessions)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp results.tsv /content/drive/MyDrive/autoresearch_results.tsv
# !cp -r checkpoints/ /content/drive/MyDrive/autoresearch_checkpoints/